In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import pandas as pd
import numpy as np

os.chdir(r"C:\Lucky\CliniScan\B13-CliniScan")

# Apni 5000 images wali CSV use karo
df = pd.read_csv("data/raw/my_5000_labels.csv")

# Class name to ID mapping
class_names = sorted(df[df['class_name'] != 'No finding']['class_name'].unique())
class_to_id = {name: idx for idx, name in enumerate(class_names)}

print("Class mapping:")
for name, idx in class_to_id.items():
    print(f"  {idx}: {name}")
print("\nTotal classes:", len(class_to_id))

Class mapping:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis

Total classes: 14


In [2]:
IMG_SIZE = 224  # tumhari images 224x224 hain
LABEL_OUTPUT = "data/processed/labels_yolo"
os.makedirs(LABEL_OUTPUT, exist_ok=True)

converted = 0
skipped = 0

for image_id in df['image_id'].unique():
    image_data = df[df['image_id'] == image_id]
    
    with open(os.path.join(LABEL_OUTPUT, image_id + ".txt"), "w") as f:
        for _, row in image_data.iterrows():
            
            # No finding skip karo
            if row['class_name'] == 'No finding':
                continue
            
            # NaN skip karo
            if pd.isna(row['x_min']):
                continue
            
            # Class ID
            class_id = class_to_id[row['class_name']]
            
            # Normalize karo 0-1 range mein
            x_center = ((row['x_min'] + row['x_max']) / 2) / IMG_SIZE
            y_center = ((row['y_min'] + row['y_max']) / 2) / IMG_SIZE
            width    = (row['x_max'] - row['x_min']) / IMG_SIZE
            height   = (row['y_max'] - row['y_min']) / IMG_SIZE
            
            # 0-1 range clip karo
            x_center = np.clip(x_center, 0, 1)
            y_center = np.clip(y_center, 0, 1)
            width    = np.clip(width, 0, 1)
            height   = np.clip(height, 0, 1)
            
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
            converted += 1

print(f"✅ Annotations converted: {converted}")
print(f"Labels saved to: {LABEL_OUTPUT}")

✅ Annotations converted: 11860
Labels saved to: data/processed/labels_yolo


In [3]:
# Kuch label files check karo
label_files = os.listdir(LABEL_OUTPUT)
print("Total label files:", len(label_files))

# Ek sample dekho
sample_file = os.path.join(LABEL_OUTPUT, label_files[0])
print("\nSample label file:", label_files[0])
with open(sample_file, 'r') as f:
    print(f.read())

Total label files: 5000

Sample label file: 0005e8e3701dfb1dd93d53e2ff537b6e.txt
7 1.000000 1.000000 1.000000 1.000000
8 1.000000 1.000000 1.000000 1.000000
6 1.000000 1.000000 1.000000 1.000000
7 1.000000 1.000000 1.000000 1.000000
4 1.000000 1.000000 1.000000 1.000000



In [4]:
# CSV mein max coordinates dekho
print("Max x_max:", df[df['class_name'] != 'No finding']['x_max'].max())
print("Max y_max:", df[df['class_name'] != 'No finding']['y_max'].max())
print("Min x_min:", df[df['class_name'] != 'No finding']['x_min'].min())
print("Min y_min:", df[df['class_name'] != 'No finding']['y_min'].min())

Max x_max: 3025.0
Max y_max: 3384.0
Min x_min: 0.0
Min y_min: 0.0


In [5]:
from PIL import Image

# Ek processed image ka size check karo
sample_img_path = "data/processed/0005e8e3701dfb1dd93d53e2ff537b6e.png"
img = Image.open(sample_img_path)
print("Processed image size:", img.size)

Processed image size: (224, 224)


In [6]:
# CSV mein is specific image ke coordinates dekho
sample_id = "0005e8e3701dfb1dd93d53e2ff537b6e"
print(df[df['image_id'] == sample_id][['image_id','x_min','y_min','x_max','y_max','class_name']])

                               image_id  x_min  y_min   x_max  y_max  \
233    0005e8e3701dfb1dd93d53e2ff537b6e  900.0  587.0  1205.0  888.0   
4548   0005e8e3701dfb1dd93d53e2ff537b6e  932.0  567.0  1197.0  896.0   
5037   0005e8e3701dfb1dd93d53e2ff537b6e  900.0  587.0  1205.0  888.0   
5604   0005e8e3701dfb1dd93d53e2ff537b6e  905.0  583.0  1203.0  890.0   
16201  0005e8e3701dfb1dd93d53e2ff537b6e  932.0  567.0  1197.0  896.0   

          class_name  
233     Lung Opacity  
4548     Nodule/Mass  
5037    Infiltration  
5604    Lung Opacity  
16201  Consolidation  


In [7]:
print(df.columns.tolist())
print(df.head(2))

['image_id', 'class_name', 'class_id', 'rad_id', 'x_min', 'y_min', 'x_max', 'y_max']
                           image_id          class_name  class_id rad_id  \
0  50a418190bc3fb1ef1633bf9678929b3          No finding        14    R11   
1  051132a778e61a86eb147c7c6f564dfe  Aortic enlargement         0    R10   

    x_min  y_min   x_max   y_max  
0     NaN    NaN     NaN     NaN  
1  1264.0  743.0  1611.0  1019.0  


In [8]:
findings = df[df['class_name'] != 'No finding']
print("x_max statistics:")
print(findings['x_max'].describe())
print("\ny_max statistics:")
print(findings['y_max'].describe())

x_max statistics:
count    11860.000000
mean      1487.764081
std        596.326092
min         71.000000
25%       1006.000000
50%       1550.000000
75%       1938.000000
max       3025.000000
Name: x_max, dtype: float64

y_max statistics:
count    11860.000000
mean      1467.184907
std        580.066962
min        122.000000
25%       1019.000000
50%       1424.000000
75%       1917.000000
max       3384.000000
Name: y_max, dtype: float64


In [1]:
import os
os.chdir(r"C:\Lucky\CliniScan\B13-CliniScan")

labels = os.listdir("data/processed/labels_yolo")
print("Total label files:", len(labels))

# Sample check karo
sample = "data/processed/labels_yolo/" + labels[0]
print("\nSample file:", labels[0])
with open(sample, 'r') as f:
    print(f.read())

Total label files: 5000

Sample file: 0005e8e3701dfb1dd93d53e2ff537b6e.txt
7 0.342611 0.240072 0.099284 0.097982
8 0.346517 0.238118 0.086263 0.107096
6 0.342611 0.240072 0.099284 0.097982
7 0.343099 0.239746 0.097005 0.099935
4 0.346517 0.238118 0.086263 0.107096


In [2]:
import os
import shutil
import random

os.chdir(r"C:\Lucky\CliniScan\B13-CliniScan")

# Folders banao
for folder in [
    "data/yolo_dataset/train/images",
    "data/yolo_dataset/train/labels",
    "data/yolo_dataset/val/images",
    "data/yolo_dataset/val/labels"
]:
    os.makedirs(folder, exist_ok=True)

# Image list banao
all_images = [f for f in os.listdir("data/processed") if f.endswith('.png')]
random.seed(42)
random.shuffle(all_images)

# 80/20 split
split = int(0.8 * len(all_images))
train_imgs = all_images[:split]
val_imgs = all_images[split:]

# Train copy karo
for img in train_imgs:
    img_id = img.replace('.png', '')
    shutil.copy(f"data/processed/{img}", f"data/yolo_dataset/train/images/{img}")
    label_src = f"data/processed/labels_yolo/{img_id}.txt"
    label_dst = f"data/yolo_dataset/train/labels/{img_id}.txt"
    if os.path.exists(label_src):
        shutil.copy(label_src, label_dst)

# Val copy karo
for img in val_imgs:
    img_id = img.replace('.png', '')
    shutil.copy(f"data/processed/{img}", f"data/yolo_dataset/val/images/{img}")
    label_src = f"data/processed/labels_yolo/{img_id}.txt"
    label_dst = f"data/yolo_dataset/val/labels/{img_id}.txt"
    if os.path.exists(label_src):
        shutil.copy(label_src, label_dst)

print("Train images:", len(os.listdir("data/yolo_dataset/train/images")))
print("Train labels:", len(os.listdir("data/yolo_dataset/train/labels")))
print("Val images:  ", len(os.listdir("data/yolo_dataset/val/images")))
print("Val labels:  ", len(os.listdir("data/yolo_dataset/val/labels")))

Train images: 4001
Train labels: 3999
Val images:   1001
Val labels:   1001


In [3]:
yaml_content = """path: data/yolo_dataset
train: train/images
val: val/images

nc: 14
names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis
"""

with open("data/yolo_dataset/dataset.yaml", "w") as f:
    f.write(yaml_content)

print("✅ dataset.yaml saved!")

# Verify
with open("data/yolo_dataset/dataset.yaml", "r") as f:
    print(f.read())

✅ dataset.yaml saved!
path: data/yolo_dataset
train: train/images
val: val/images

nc: 14
names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis

